# RumourEval Context-Error Sensitivity MVP Runner

이 노트북은 RumourEval 2019 Task A context-error sensitivity 1차 실험을 JupyterLab에서 바로 실행하기 위한 runner입니다.

기본값으로 `Run All`을 누르면 다음까지 실행됩니다.

1. repo root 자동 탐색
2. dependency 확인
3. unit test
4. RumourEval raw data 다운로드
5. reply target parsing
6. C0-C4 context variant 생성
7. majority baseline 평가
8. Qwen2.5-0.5B smoke inference, 기본 100 variant rows

Full dev/test run은 아래 설정 셀에서 `RUN_FULL_DEV` 또는 `RUN_TEST_ONCE`만 바꾸면 됩니다.

In [ ]:
from pathlib import Path
import csv
import importlib.util
import json
import os
import subprocess
import sys
import time
from html import escape

from IPython.display import HTML, Markdown, display


# If auto-detection fails, set this manually, e.g. "/home/ubuntu/nlp-project-2026".
MANUAL_PROJECT_ROOT = os.environ.get("NLP_PROJECT_ROOT", "")

REPO_MARKERS = ("configs/experiment_mvp.yaml", "src")
GITHUB_REPO_URL = "https://github.com/angellashin/nlp-project-2026.git"


def is_repo_root(path):
    path = Path(path).expanduser().resolve()
    return all((path / marker).exists() for marker in REPO_MARKERS)


def git_repo_root(start):
    try:
        result = subprocess.run(
            ["git", "-C", str(start), "rev-parse", "--show-toplevel"],
            capture_output=True,
            text=True,
            timeout=5,
        )
    except Exception:
        return None
    if result.returncode != 0:
        return None
    candidate = Path(result.stdout.strip())
    return candidate if is_repo_root(candidate) else None


def nearby_dirs(base, max_depth=3):
    base = Path(base).expanduser()
    if not base.exists():
        return
    queue = [(base.resolve(), 0)]
    seen = set()
    skip = {".cache", ".conda", ".git", ".ipynb_checkpoints", "data", "results", "__pycache__"}
    while queue:
        path, depth = queue.pop(0)
        if path in seen:
            continue
        seen.add(path)
        yield path
        if depth >= max_depth:
            continue
        try:
            children = [child for child in path.iterdir() if child.is_dir() and child.name not in skip]
        except OSError:
            continue
        queue.extend((child, depth + 1) for child in children)


def find_repo_root(start=None):
    starts = []
    if start:
        starts.append(Path(start).expanduser())
    starts.extend([Path.cwd(), Path.home(), Path("/home/ubuntu"), Path("/workspace")])

    for raw_start in starts:
        if not raw_start.exists():
            continue
        start_path = raw_start.resolve()
        for path in [start_path, *start_path.parents]:
            if is_repo_root(path):
                return path
        git_root = git_repo_root(start_path)
        if git_root:
            return git_root

    common_names = ["nlp-project-2026", "NLP project"]
    for raw_start in starts:
        if not raw_start.exists():
            continue
        for name in common_names:
            candidate = raw_start / name
            if candidate.exists() and is_repo_root(candidate):
                return candidate.resolve()

    for raw_start in starts:
        if not raw_start.exists():
            continue
        for candidate in nearby_dirs(raw_start, max_depth=3):
            if is_repo_root(candidate):
                return candidate

    message = f"""
Could not find the nlp-project-2026 repo root.

Most likely, Jupyter opened this notebook while the kernel cwd is outside the repo,
or only the .ipynb file was uploaded without the rest of the GitHub repo.

Fix in a Jupyter terminal:
  cd /home/ubuntu
  git clone {GITHUB_REPO_URL}

Then open:
  /home/ubuntu/nlp-project-2026/notebooks/rumoureval_mvp_runner.ipynb

If the repo already exists somewhere else, set this in the first code cell:
  MANUAL_PROJECT_ROOT = "/absolute/path/to/nlp-project-2026"

Current kernel cwd: {Path.cwd()}
"""
    raise RuntimeError(message)


PROJECT_ROOT = find_repo_root(MANUAL_PROJECT_ROOT or None)
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"Python = {sys.executable}")

In [ ]:
def run(args, check=True):
    args = [str(arg) for arg in args]
    print("\n$ " + " ".join(args), flush=True)
    started = time.time()
    result = subprocess.run(args, cwd=PROJECT_ROOT, text=True)
    elapsed = time.time() - started
    print(f"[exit={result.returncode} | {elapsed:.1f}s]", flush=True)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {' '.join(args)}")
    return result


def read_json(path):
    path = PROJECT_ROOT / path if not isinstance(path, Path) else path
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def read_jsonl(path, limit=None):
    path = PROJECT_ROOT / path if not isinstance(path, Path) else path
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if limit is not None and i >= limit:
                break
            rows.append(json.loads(line))
    return rows


def show_json(path):
    payload = read_json(path)
    display(Markdown(f"```json\n{json.dumps(payload, indent=2, ensure_ascii=False)}\n```"))


def show_csv(path, max_rows=20):
    path = PROJECT_ROOT / path if not isinstance(path, Path) else path
    if not path.exists():
        print(f"missing: {path}")
        return
    try:
        import pandas as pd

        display(pd.read_csv(path).head(max_rows))
        return
    except Exception:
        pass

    with path.open("r", encoding="utf-8", newline="") as f:
        rows = list(csv.DictReader(f))[:max_rows]
    if not rows:
        print(f"empty csv: {path}")
        return
    headers = list(rows[0].keys())
    table = ["<table><thead><tr>"]
    table.extend(f"<th>{escape(header)}</th>" for header in headers)
    table.append("</tr></thead><tbody>")
    for row in rows:
        table.append("<tr>")
        table.extend(f"<td>{escape(str(row.get(header, '')))}</td>" for header in headers)
        table.append("</tr>")
    table.append("</tbody></table>")
    display(HTML("".join(table)))

## 0. Run Settings

기본값은 빠른 smoke run입니다. Full dev run을 돌릴 준비가 되면 `RUN_FULL_DEV = True`로 바꾸고 이 셀부터 다시 실행하면 됩니다. Test set은 prompt/context rule이 freeze된 뒤에만 `RUN_TEST_ONCE = True`로 바꾸세요.

In [ ]:
SPLITS = ["train", "dev", "test"]
SEED = 461

INSTALL_MISSING_DEPS = True
RUN_DOWNLOAD = True

RUN_SMOKE_QWEN = True
SMOKE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
SMOKE_RUN_ID = "dev_qwen25_05b_smoke"
SMOKE_LIMIT = 100  # 20 reply targets x 5 conditions, because variants are emitted target-by-target

RUN_FULL_DEV = False
FULL_DEV_MODELS = [
    ("Qwen/Qwen2.5-0.5B-Instruct", "dev_qwen25_05b"),
    ("Qwen/Qwen2.5-1.5B-Instruct", "dev_qwen25_15b"),
]

RUN_TEST_ONCE = False
TEST_MODELS = FULL_DEV_MODELS

DTYPE = "float16"  # L4에서 현실적인 기본값. 필요하면 "bfloat16" 또는 "auto"로 변경.
MAX_NEW_TOKENS = 8
PROMPT_VERSION = "qwen_mvp_v2"

print("Configured smoke run:", RUN_SMOKE_QWEN, SMOKE_MODEL, SMOKE_LIMIT)
print("Configured full dev run:", RUN_FULL_DEV)
print("Configured test run:", RUN_TEST_ONCE)

## 1. Environment Check

In [ ]:
run([sys.executable, "--version"], check=False)
run(["nvidia-smi"], check=False)

In [ ]:
required_modules = ["torch", "transformers", "yaml", "sentencepiece"]
missing = [name for name in required_modules if importlib.util.find_spec(name) is None]
print("missing modules:", missing)

if missing and INSTALL_MISSING_DEPS:
    run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"])
elif missing:
    print("Some dependencies are missing. Set INSTALL_MISSING_DEPS=True or install requirements manually.")
else:
    print("Core dependencies are available.")

## 2. Local Code Validation

GPU inference 전에 parser/context/evaluation unit tests를 먼저 확인합니다.

In [ ]:
run([sys.executable, "-m", "unittest", "discover", "-s", "tests"])

## 3. Download and Parse RumourEval 2019

Raw data는 `data/raw/rumoureval2019/` 아래에 저장됩니다. 이 폴더는 git에 올라가지 않습니다.

In [ ]:
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "rumoureval2019"
archive = RAW_DIR / "rumoureval2019.tar.bz2"
extracted_marker = RAW_DIR / "rumoureval2019" / "rumoureval-2019-training-data.zip"

if RUN_DOWNLOAD and not archive.exists() and not extracted_marker.exists():
    run([sys.executable, "-m", "src.data.download_rumoureval", "--out-dir", RAW_DIR])
else:
    print(f"Raw data already present or download disabled: {RAW_DIR}")

In [ ]:
run([
    sys.executable,
    "-m",
    "src.data.parse_rumoureval",
    "--raw-dir",
    "data/raw/rumoureval2019",
    "--out-dir",
    "data/processed",
])

show_json("data/processed/dataset_summary.json")

## 4. Build C0-C4 Context Variants

각 reply target마다 `reply_only`, `useful`, `irrelevant`, `conflicting`, `mixed` 5개 condition row를 생성합니다.

In [ ]:
run([
    sys.executable,
    "-m",
    "src.data.build_context_variants",
    "--processed-dir",
    "data/processed",
    "--out-dir",
    "data/variants",
    "--splits",
    *SPLITS,
    "--seed",
    SEED,
])

show_json("data/variants/coverage_summary.json")

In [ ]:
dev_variants = read_jsonl("data/variants/dev_context_variants.jsonl", limit=10)
print("First 10 dev variant ids:")
for row in dev_variants:
    print(row["example_id"], row["label"], row["condition"], row["construction_notes"])

print("\nPrompt preview:")
print(dev_variants[0]["prompt_text"][:2000])

## 5. Majority Baseline

이 baseline은 모든 예측을 majority class인 `comment`로 두는 sanity check입니다. Accuracy는 높고 macro-F1은 낮아야 정상입니다.

In [ ]:
run([
    sys.executable,
    "-m",
    "src.experiments.run_majority_baseline",
    "--variants",
    "data/variants/dev_context_variants.jsonl",
    "--out-dir",
    "results/runs/dev_majority_baseline",
])

run([
    sys.executable,
    "-m",
    "src.experiments.evaluate",
    "--predictions",
    "results/runs/dev_majority_baseline/predictions.jsonl",
    "--out-dir",
    "results/tables/dev_majority_baseline",
])

display(Markdown("### Majority Baseline Summary"))
show_csv("results/tables/dev_majority_baseline/summary_metrics.csv")
display(Markdown("### Majority Baseline Per-Class Metrics"))
show_csv("results/tables/dev_majority_baseline/per_class_metrics.csv")
display(Markdown("### Majority Baseline By Platform"))
show_csv("results/tables/dev_majority_baseline/summary_by_platform.csv")
display(Markdown("### Majority Baseline Predicted Label Distribution"))
show_csv("results/tables/dev_majority_baseline/predicted_label_distribution.csv")

## 6. Qwen Smoke Run

기본 smoke run은 dev variant 중 앞 100개 row만 실행합니다. Variant가 target별로 5개씩 생성되므로 대략 20개 reply target의 C0-C4를 확인하는 셈입니다.

In [ ]:
if RUN_SMOKE_QWEN:
    run([
        sys.executable,
        "-m",
        "src.experiments.run_prompting",
        "--variants",
        "data/variants/dev_context_variants.jsonl",
        "--out-dir",
        f"results/runs/{SMOKE_RUN_ID}",
        "--model",
        SMOKE_MODEL,
        "--limit",
        SMOKE_LIMIT,
        "--dtype",
        DTYPE,
        "--max-new-tokens",
        MAX_NEW_TOKENS,
        "--prompt-version",
        PROMPT_VERSION,
    ])

    run([
        sys.executable,
        "-m",
        "src.experiments.evaluate",
        "--predictions",
        f"results/runs/{SMOKE_RUN_ID}/predictions.jsonl",
        "--out-dir",
        f"results/tables/{SMOKE_RUN_ID}",
    ])

    display(Markdown("### Qwen Smoke Run Metadata"))
    show_json(f"results/runs/{SMOKE_RUN_ID}/run_metadata.json")
    display(Markdown("### Qwen Smoke Summary"))
    show_csv(f"results/tables/{SMOKE_RUN_ID}/summary_metrics.csv")
    display(Markdown("### Qwen Smoke Context Gaps"))
    show_csv(f"results/tables/{SMOKE_RUN_ID}/context_gaps.csv")
    display(Markdown("### Qwen Smoke By Platform"))
    show_csv(f"results/tables/{SMOKE_RUN_ID}/summary_by_platform.csv")
    display(Markdown("### Qwen Smoke By Validity Subset"))
    show_csv(f"results/tables/{SMOKE_RUN_ID}/summary_by_validity_subset.csv")
    display(Markdown("### Qwen Smoke Predicted Label Distribution"))
    show_csv(f"results/tables/{SMOKE_RUN_ID}/predicted_label_distribution.csv")
else:
    print("RUN_SMOKE_QWEN=False, skipping smoke inference.")

## 7. Full Dev Run

Smoke run의 invalid rate와 runtime이 괜찮으면 설정 셀에서 `RUN_FULL_DEV = True`로 바꾼 뒤 이 셀부터 실행하세요. 이 단계가 실제 main result matrix입니다.

In [ ]:
if RUN_FULL_DEV:
    for model_name, run_id in FULL_DEV_MODELS:
        run([
            sys.executable,
            "-m",
            "src.experiments.run_prompting",
            "--variants",
            "data/variants/dev_context_variants.jsonl",
            "--out-dir",
            f"results/runs/{run_id}",
            "--model",
            model_name,
            "--dtype",
            DTYPE,
            "--max-new-tokens",
            MAX_NEW_TOKENS,
            "--prompt-version",
            PROMPT_VERSION,
        ])

        run([
            sys.executable,
            "-m",
            "src.experiments.evaluate",
            "--predictions",
            f"results/runs/{run_id}/predictions.jsonl",
            "--out-dir",
            f"results/tables/{run_id}",
        ])

        display(Markdown(f"### {run_id} Summary"))
        show_json(f"results/runs/{run_id}/run_metadata.json")
        show_csv(f"results/tables/{run_id}/summary_metrics.csv")
        display(Markdown(f"### {run_id} Context Gaps"))
        show_csv(f"results/tables/{run_id}/context_gaps.csv")
        display(Markdown(f"### {run_id} By Platform"))
        show_csv(f"results/tables/{run_id}/summary_by_platform.csv")
        display(Markdown(f"### {run_id} By Depth Bucket"))
        show_csv(f"results/tables/{run_id}/summary_by_depth_bucket.csv")
        display(Markdown(f"### {run_id} By Context Source"))
        show_csv(f"results/tables/{run_id}/summary_by_context_source.csv")
        display(Markdown(f"### {run_id} By Validity Subset"))
        show_csv(f"results/tables/{run_id}/summary_by_validity_subset.csv")
        display(Markdown(f"### {run_id} Predicted Label Distribution"))
        show_csv(f"results/tables/{run_id}/predicted_label_distribution.csv")
        display(Markdown(f"### {run_id} Paired Flip Rates"))
        show_csv(f"results/tables/{run_id}/paired_flip_rates.csv")
else:
    print("RUN_FULL_DEV=False. After smoke validation, set RUN_FULL_DEV=True to run both Qwen models on full dev.")

## 8. Available Result Summary

지금까지 생성된 `results/tables/*/summary_metrics.csv`를 모아서 확인합니다.

In [ ]:
summary_files = sorted((PROJECT_ROOT / "results" / "tables").glob("*/summary_metrics.csv"))
print(f"found {len(summary_files)} summary files")

all_summary_rows = []
for summary_path in summary_files:
    with summary_path.open("r", encoding="utf-8", newline="") as f:
        for row in csv.DictReader(f):
            row = dict(row)
            row["run_id"] = summary_path.parent.name
            all_summary_rows.append(row)

if all_summary_rows:
    headers = ["run_id", "model", "condition", "n", "accuracy", "macro_f1", "invalid_rate"]
    table = ["<table><thead><tr>"]
    table.extend(f"<th>{escape(header)}</th>" for header in headers)
    table.append("</tr></thead><tbody>")
    for row in all_summary_rows:
        table.append("<tr>")
        table.extend(f"<td>{escape(str(row.get(header, '')))}</td>" for header in headers)
        table.append("</tr>")
    table.append("</tbody></table>")
    display(HTML("".join(table)))
else:
    print("No result summaries yet.")

## 9. Error Analysis Case Extraction

Full dev run이 있으면 full dev 결과를, 없으면 smoke 결과를 사용해서 prediction flip 사례를 뽑습니다.

In [ ]:
preferred_error_runs = ["dev_qwen25_05b", "dev_qwen25_15b", SMOKE_RUN_ID]
error_run_id = next(
    (run_id for run_id in preferred_error_runs if (PROJECT_ROOT / "results" / "runs" / run_id / "predictions.jsonl").exists()),
    None,
)

if error_run_id:
    error_out = f"results/tables/{error_run_id}_error_cases.csv"
    run([
        sys.executable,
        "-m",
        "src.analysis.error_analysis",
        "--predictions",
        f"results/runs/{error_run_id}/predictions.jsonl",
        "--variants",
        "data/variants/dev_context_variants.jsonl",
        "--out",
        error_out,
        "--limit",
        50,
    ])
    display(Markdown(f"### Error cases from `{error_run_id}`"))
    show_csv(error_out, max_rows=20)
else:
    print("No Qwen predictions found yet. Run the smoke or full dev inference first.")

## 10. Test Run After Rules Are Frozen

중요: dev에서 prompt/context construction rule을 확정한 뒤에만 `RUN_TEST_ONCE = True`로 바꾸세요. Test set은 최종 평가용으로 한 번만 쓰는 것을 원칙으로 합니다.

In [ ]:
if RUN_TEST_ONCE:
    for model_name, dev_run_id in TEST_MODELS:
        test_run_id = dev_run_id.replace("dev_", "test_", 1)
        run([
            sys.executable,
            "-m",
            "src.experiments.run_prompting",
            "--variants",
            "data/variants/test_context_variants.jsonl",
            "--out-dir",
            f"results/runs/{test_run_id}",
            "--model",
            model_name,
            "--dtype",
            DTYPE,
            "--max-new-tokens",
            MAX_NEW_TOKENS,
            "--prompt-version",
            PROMPT_VERSION,
        ])

        run([
            sys.executable,
            "-m",
            "src.experiments.evaluate",
            "--predictions",
            f"results/runs/{test_run_id}/predictions.jsonl",
            "--out-dir",
            f"results/tables/{test_run_id}",
        ])

        display(Markdown(f"### {test_run_id} Summary"))
        show_csv(f"results/tables/{test_run_id}/summary_metrics.csv")
else:
    print("RUN_TEST_ONCE=False. Keep it false until dev rules are frozen.")

## Output Checklist

주요 산출물 위치:

- parsed targets: `data/processed/reply_targets_{train,dev,test}.jsonl`
- context variants: `data/variants/{split}_context_variants.jsonl`
- predictions: `results/runs/{run_id}/predictions.jsonl`
- run metadata: `results/runs/{run_id}/run_metadata.json`
- metrics: `results/tables/{run_id}/summary_metrics.csv`
- per-class metrics: `results/tables/{run_id}/per_class_metrics.csv`
- context gaps: `results/tables/{run_id}/context_gaps.csv`
- platform summary: `results/tables/{run_id}/summary_by_platform.csv`
- platform context gaps: `results/tables/{run_id}/context_gaps_by_platform.csv`
- depth/context/validity summaries: `results/tables/{run_id}/summary_by_*.csv`
- predicted label distribution: `results/tables/{run_id}/predicted_label_distribution.csv`
- paired flips: `results/tables/{run_id}/paired_flip_rates.csv`, `paired_flip_cases.csv`
- error-analysis samples: `results/tables/{run_id}_error_cases.csv`